# Component 3 — Exploratory Data Analysis & Data Preparation
## End-to-End Traffic Volume Forecasting

**Project objective:** Prepare historical traffic and weather data for predictive modelling, perform exploratory analysis, engineer temporal features, and document the resulting dataset.

**Dataset:** `integrated_traffic_dataset.csv`


## 1. Import Essential Libraries

We import Pandas and NumPy for data handling, Matplotlib/Seaborn for visualization, and warnings to keep the notebook output clean.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams.update({"figure.max_open_warning": 0, "font.size": 11})


## 2. Load the Integrated Dataset

The integrated dataset contains historical vehicle counts together with junction, date/time, and localized weather information.


In [ ]:
df = pd.read_csv("integrated_traffic_dataset.csv")

print("Dataset Shape:", df.shape)
display(df.head())
display(df.info())


## 3. Initial Data Quality Inspection

Before modelling, we check duplicate records, missing values, data types, and basic numerical statistics.


In [ ]:
print("Missing values:")
display(df.isna().sum().to_frame("Missing Count"))

print("Duplicate rows:", df.duplicated().sum())

print("\nData types:")
display(df.dtypes.to_frame("Data Type"))

print("\nNumerical summary:")
display(df.describe(include="all").T)


## 4. DateTime Conversion & Temporal Feature Engineering

Traffic is strongly time-dependent. We convert `DateTime` and derive:
- Hour
- Day of Week
- Month
- Day
- Year
- Weekend indicator
- Peak-hour indicator


In [ ]:
df["DateTime"] = pd.to_datetime(
    df["DateTime"], format="mixed", dayfirst=True, errors="coerce"
)

df = df.drop_duplicates().reset_index(drop=True)

df["Hour"] = df["DateTime"].dt.hour
df["DayOfWeek"] = df["DateTime"].dt.dayofweek
df["Month"] = df["DateTime"].dt.month
df["Day"] = df["DateTime"].dt.day
df["Year"] = df["DateTime"].dt.year
df["Is_Weekend"] = df["weekend"].astype(int)

# Morning/evening traffic peak indicator
df["Is_Peak_Hour"] = df["Hour"].isin([7, 8, 9, 17, 18, 19]).astype(int)

display(df[[
    "DateTime", "Hour", "DayOfWeek", "Month",
    "Is_Weekend", "Is_Peak_Hour"
]].head())


## 5. Missing-Value Treatment

Numerical missing values are filled using the median of each numerical column. Records without a usable DateTime or target value are removed because those rows cannot be used reliably for time-based traffic modelling.


In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns

for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

df = df.dropna(subset=["DateTime", "Vehicles"]).reset_index(drop=True)

print("Remaining missing values:", int(df.isna().sum().sum()))
print("Prepared dataset shape:", df.shape)


## 6. Hourly Traffic Trend

This visualization identifies recurring daily traffic patterns and helps locate high-volume periods.


In [ ]:
hourly = df.groupby("Hour", as_index=False)["Vehicles"].mean()

plt.figure(figsize=(12, 5))
plt.plot(hourly["Hour"], hourly["Vehicles"], marker="o")
plt.title("Average Hourly Traffic Volume")
plt.xlabel("Hour of Day")
plt.ylabel("Average Vehicles")
plt.xticks(range(24))
plt.tight_layout()
plt.show()


## 7. Traffic Pattern Across Junctions

Different road junctions can have different traffic levels. This visualization compares their hourly patterns.


In [ ]:
plt.figure(figsize=(12, 5))

for junction, group in df.groupby("Junction"):
    temp = group.groupby("Hour")["Vehicles"].mean()
    plt.plot(temp.index, temp.values, marker="o", label=f"Junction {junction}")

plt.title("Hourly Traffic Pattern Across Junctions")
plt.xlabel("Hour")
plt.ylabel("Average Vehicles")
plt.xticks(range(24))
plt.legend()
plt.tight_layout()
plt.show()


## 8. Junction-Wise Traffic Distribution

A boxplot helps compare the spread and potential outliers of traffic volume at each junction.


In [ ]:
plt.figure(figsize=(9, 5))
sns.boxplot(data=df, x="Junction", y="Vehicles")
plt.title("Traffic Volume Distribution by Junction")
plt.xlabel("Junction")
plt.ylabel("Vehicles")
plt.tight_layout()
plt.show()


## 9. Weekday vs Weekend Traffic


In [ ]:
weekday = df.loc[df["Is_Weekend"] == 0, "Vehicles"]
weekend = df.loc[df["Is_Weekend"] == 1, "Vehicles"]

plt.figure(figsize=(8, 5))
plt.boxplot([weekday, weekend], labels=["Weekday", "Weekend"])
plt.title("Traffic Volume: Weekday vs Weekend")
plt.ylabel("Vehicles")
plt.tight_layout()
plt.show()


## 10. Monthly Traffic Trend


In [ ]:
monthly = df.groupby("Month", as_index=False)["Vehicles"].mean()

plt.figure(figsize=(9, 5))
plt.bar(monthly["Month"], monthly["Vehicles"])
plt.title("Average Monthly Traffic Volume")
plt.xlabel("Month")
plt.ylabel("Average Vehicles")
plt.tight_layout()
plt.show()


## 11. Peak-Hour Identification


In [ ]:
peak_hours = (
    df.groupby("Hour")["Vehicles"]
      .mean()
      .sort_values(ascending=False)
      .head(5)
      .reset_index()
)

display(peak_hours)

plt.figure(figsize=(8, 5))
plt.bar(peak_hours["Hour"].astype(str), peak_hours["Vehicles"])
plt.title("Top 5 Peak Traffic Hours")
plt.xlabel("Hour")
plt.ylabel("Average Vehicles")
plt.tight_layout()
plt.show()


## 12. Weather vs Traffic Analysis


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(
    df["temperature_2m (°C)"], df["Vehicles"], alpha=0.25, s=8
)
axes[0].set_title("Temperature vs Traffic")
axes[0].set_xlabel("Temperature (°C)")
axes[0].set_ylabel("Vehicles")

axes[1].scatter(
    df["precipitation (mm)"], df["Vehicles"], alpha=0.25, s=8
)
axes[1].set_title("Precipitation vs Traffic")
axes[1].set_xlabel("Precipitation (mm)")
axes[1].set_ylabel("Vehicles")

plt.tight_layout()
plt.show()


## 13. Correlation Analysis

The correlation matrix summarizes pairwise linear relationships. Correlation should be interpreted as association, not causation.


In [ ]:
corr_cols = [
    "Vehicles",
    "temperature_2m (°C)",
    "relative_humidity_2m (%)",
    "precipitation (mm)",
    "wind_speed_10m (km/h)",
    "Hour",
    "DayOfWeek",
    "Month",
    "Is_Weekend"
]

corr = df[corr_cols].corr()

plt.figure(figsize=(11, 8))
sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    linewidths=0.5
)
plt.title("Correlation Matrix: Traffic, Temporal and Weather Variables")
plt.tight_layout()
plt.show()


## 14. Junction × Hour Heatmap


In [ ]:
junction_hour = df.pivot_table(
    index="Junction",
    columns="Hour",
    values="Vehicles",
    aggfunc="mean"
)

plt.figure(figsize=(13, 5))
sns.heatmap(junction_hour, cmap="YlOrRd")
plt.title("Average Traffic Volume by Junction and Hour")
plt.xlabel("Hour")
plt.ylabel("Junction")
plt.tight_layout()
plt.show()


## 15. Final Prepared Dataset

The prepared file is saved for use by the model-development notebook.


In [ ]:
df.to_csv("traffic_preprocessed.csv", index=False)

print("Final prepared dataset shape:", df.shape)
print("Saved as: traffic_preprocessed.csv")
display(df.head())
